# INF201 - Lecture 6
12th Oct 2026

## Topics
* Deliverables
* A reminder of GitHub collaborative work flow
* Regular expressions

## Deliverables
* The first deliverable must be submitted by Friday at 16
* As the exercise classes are no longer mandatory, they are less full, so feel free attend classes other<br>
than the one you are signed up for if you want more support, or maybe just a place to work
* I have created a new set of groups on Canvas under the heading Deliverables
    * These are called something like this: G1_4
    * The deliverables are group assignments, meaning that one of you submit on behalf of both/all of you
    * Communicate with each other to make sure the assignment is submitted on time!
    * Check that you have uploaded the files correctly!
* Remember that we asked you to make a new virtual environment (venv) for each of the deliverables
    * To show that you have done so, include a requirements.txt (created by `pip freeze`) in your repo
* Please use English for these assignments!

## A quick reminder of how to collaborate in GitHub
* Create the repository <i>before</i> you start working on the code: This way everything gets tracked
* Use branches rather than working directly on main
* This will give fewer conflicts than if you are both editing the same document on main
* Merge the branch back to main when it is working well
* Remember to commit, pull and push <i>often</i>: This keeps your local repo up to date with the remote

## Regular expressions

### An example for motivation
* You are given a text file and asked to extract all email addresses mentioned in the file.
* How would you do this in Python?

In [ ]:
text = open("unstructured_leads.txt").read()  # File generated by Gemini
print(text)

1. Read file (line by line?)
2. Split the string into "words"

In [ ]:
# words = text.split()
# print(words)

with open("unstructured_leads.txt") as f:
    for line in f:
        words = line.split()
        print(words)

3. Split gave us the "words" and removed the spaced between them, but some punctuation still remains and needs to be removed:  `[`, `]`, `(`, `)`, `.`, `:`, `!`
4. We are looking for email addresses, so we only keep to keep "words" with `@` in them 

In [ ]:
words = text.split()
for word in words:
    if "@" in word:
        cleaned = word.strip(".:!()[]")
        print(cleaned)

# with open("unstructured_leads.txt") as f:
#     for line in f:
#         words = line.split()
#         for word in words:
#             if "@" in word:
#                 cleaned = word.strip(".()[]")
#                 print(cleaned)

5. This looks nice! However, we should also verify that these "words" are formatted like an email address, in case we want to use the script on a different file

In [ ]:
emails_found = []
words = text.split()
for word in words:
    if "@" in word:
        cleaned = word.strip(".:!()[]")
        if cleaned.count("@") == 1:
            parts = cleaned.split("@")
            if parts[0] and "." in parts[1]:
                # Check that the first part is non-empty and that the second part (domain) has at least one "."
                emails_found.append(cleaned)
emails_found

# emails_found = []
# with open("unstructured_leads.txt") as f:
#     for line in f:
#         words = line.split()
#         for word in words:
#             if "@" in word:
#                 cleaned = word.strip(".()[]")
#                 if cleaned.count("@") == 1:
#                     parts = cleaned.split("@")
#                     if parts[0] and "." in parts[1]:
#                         emails_found.append(cleaned)
# emails_found

* Nice! We solved the problem with 9 lines of code (10 if we include reading the file).
* Now, could we easily modify the code to look for phone numbers instead?

In [ ]:
phones_found = []
words = text.split()
for word in words:
    # Clean off trailing sentence punctuation
    cleaned = word.strip(".:!()[]")

    # Count actual digits
    digit_count = sum(c.isdigit() for c in cleaned)

    # Check if all characters are valid phone symbols (+, -, ., digits)
    valid_chars = all(c.isdigit() or c in "+-." for c in cleaned)

    # Require at least 10 digits and valid symbols
    if digit_count >= 10 and valid_chars:
        phones_found.append(cleaned)
phones_found

* We got there, but had to re-write most of the code
* Let's find out how we can solve these problems in a more compact and flexible way
* Using regular expressions, each of these problems can be solved with one line of code

In [ ]:
import re

emails_regex = re.findall(r"[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}", text)
emails_regex

In [ ]:
phones_regex = re.findall(r"\+?\d[\d.-]{8,}\d", text)
phones_regex

### What are regular expressions?
> Regular expression is a very powerful way of processing text while looking for recurring patterns
* From chapter 31 in [Advanced Guide to Python 3 Programming](https://link.springer.com/book/10.1007/978-3-031-40336-1)
* Often just called regex
    * Strickly speaking regular expression is the concept in theoretical computer science
    * Regex is the practical implementation in software
* [Introduction](https://docs.python.org/3/howto/regex.html) to using the `re` package in Python for regular expressions
* [regex101](https://regex101.com/) allows you to test out regular expressions
* If you open a text file in VS Code and press `Ctrl`+`F`/`Cmnd`+`F` and toggle the Regex option, you can highlight which parts of the text matches the specified pattern. 

### The basics for forming a regular expression
* Define a pattern
* Search for that pattern in a string
* Optionally, match groups within a string
* Optionally, create new string by replacing the pattern

### Some simple Python examples
* First, we will use `re.search()` to look for a word in a string
* The first argument is the pattern we are looking for, and the second argument is the string

In [ ]:
re.search("short", "This is a short line of text")

* This returned a [re.Match object](https://docs.python.org/3/library/re.html#match-objects)
* This will among other things give us the locations of the match in the string

In [ ]:
m = re.search("short", "This is a short line of text")
print("String:", m.string)
print("Pattern:", m.re)
print("Match:", m.group())
print("Span:", m.span())
print("Start:", m.start())
print("Stop:", m.end())

In [ ]:
m = re.search("long", "This is a short line of text")
print(m)

In [ ]:
re.search("is", "This is a short line of text")

* Note than when we searched for "is", we got the first instance of these two letters together,<br>
in other words the second half of the word "This"
* `re.match()` returns the first match
* We can find all matches by using `re.findall()`

In [ ]:
re.findall("is", "This is a short line of text")

In [ ]:
re.sub("short", "long", "This is a short line of text")

### More advanced use
* By creating more flexible patterns, this becomes much more interesting!
* The `re` packages has several useful functions, for example:
    * `re.search(pattern, string)` finds first match of pattern in string
    * `re.match(pattern, string)` as `search()`, but match <i>only</i> at beginning of string
    * `re.findall(pattern, string)` finds all matches of pattern in string
    * `re.finditer(pattern, string)` iterate over all matches in string, returning match objects
    * `re.split(pattern, string)` split string on pattern and return list of parts
    * `re.sub(pattern, replacement, string)` replaces all occurances of pattern in string with replacement
    * `re.compile(pattern)` returns regular expression object, improves efficiency if same regex is used many times
* So far we have used regular strings as patterns, but typically you would want to use <i>raw</i> string
    * Regexes often contain backslashes, and using raw strings makes sure this is interpreted correctly by Python

In [ ]:
r"This is a raw string. \n We use it to deal with \ in strings."

In [ ]:
normal = "This is a string \n with a newline character inside."
print(normal)

In [ ]:
raw = r"This is a string \n with a newline character inside."
print(raw)

* Regexes are more powerful when we use flexible patterns
* Let's look for all words ending with the letter `e`

In [ ]:
re.findall(r"\w*e\b", "I can see the red rose and the purple lilac.")

* Break-down of the pattern:
    * `\w`: Matches any character that can be part of a word (letters, numbers, underscore)
    * `*`: Match this 0 or more times
    * `e`: The lower-case letter e
    * `\b`: Matches any word-boundary character

* Now, let's find all the words that are at least four letters long

In [ ]:
re.findall(r"\w{4,}", "I can see the red rose and the purple lilac.")

* `{4,}` means: previous symbol repeated at least four times
* More generally: `{m,n}` means at least `m` repetitions and at most `n` repetitions

* Now we look for three-word sequences beginning with "the"

In [ ]:
re.findall(r"the\s+\w+\s+\w+\b", "I can see the red rose and the purple lilac.")

* Break-down of the pattern:
    * `the`: The word "the" (all lower-case)
    * `\s+`: At least one occurance (`+`) of a whitespace (`\s`)
    * `\w+`: At least one occurance (`+`) of any word-characters
    * `\b`: A word-boundary characters

### Grouping in regular expressions
* Use `()` to extract sub-components of a string
* Let's extract colour and plant names from the same string as before

In [ ]:
re.findall(r"the\s+(\w+)\s+(\w+)\b", "I can see the red rose and the purple lilac.")

* For each match of the entire pattern (e.g. "the red rose") we now get returned the parts in parentheses
* We can access the parts in the groups in the replacement strings
    * `\1` is the first group, `\2` the second

In [ ]:
re.sub(r"the\s+(\w+)\s+(\w+)\b", r"a \2 [\1]", "I can see the red rose and the purple lilac.")

* We can also name the groups
    * Syntax: `(?P<colour>\w+)` will give the name `colour` to the group captured with `(\w+)`

In [ ]:
re.sub(r"the\s+(?P<colour>\w+)\s+(?P<plant>\w+)\b",
       r"a \g<plant> [\g<colour>]",
       "I can see the red rose and the purple lilac.")

* Using flexible patterns and grouping makes the output of functions like `match()` much more interesting

In [ ]:
m = re.search(r"the\s+(\w+)\s+(\w+)\b",
              "I can see the red rose and the purple lilac.")
print("Match object:", m)
print("Match string:", m.group())
print("Groups      :", m.groups())

In [ ]:
m = re.search(r"the\s+(?P<color>\w+)\s+(?P<plant>\w+)\b",
              "I can see the red rose and the purple lilac.")
print("Match object:", m)
print("Match string:", m.group())
print("Groups      :", m.groups())
print("Group dict  :", m.groupdict())

* Let's try this with a text-file
* By using named groups, we get a dictionary of groups

In [ ]:
pattern = r"(?P<Name>.+);(?P<Age>\d+);(?P<Phone>.+)"
# pattern = re.compile(r"(?P<Name>.+);(?P<Age>\d+);(?P<Phone>.+)")
with open("students_with_header.txt") as f:
    f.readline()
    students = [re.search(pattern, line).groupdict() for line in f]
students

In [ ]:
re.compile(r"(?P<Name>.+);(?P<Age>\d+);(?P<Phone>.+)")

* Now a more advanced example:
    * We can refer to a group inside a search pattern
    * Use this to look for repetitions of words (which probably are the result of a typo)

In [ ]:
re.search(r"\b(\w+)\s+\1\b", "I can see the the red rose rose and the purple lilac.")

* Break-down of this pattern:
    * `\b`: a word boundary (we must start at the beginning of a word)
    * `(\w+)`: matches a non-empty word and collects it as into a group
    * `\s+`: at least one whitespace (if there is a comma or similar between words, repeated words are probably ok)
    * `\1`: that the word collected in the group must occur here again
    * `\b`: a word boundary
* Use information from the match object to mark location of problem.

In [ ]:
text = "I can see the the red rose rose and the purple lilac."
m = re.search(r"\b(\w+)\s+\1\b", text)
print(text)
print('-' * m.start() + '^' * (m.end() - m.start()))

* We can use `finditer()` to get one match at a time

In [ ]:
for m in re.finditer(r"\b(\w+)\s+\1\b",
                    "I can see the the red rose rose and the purple lilac."):
    print(m)

In [ ]:
text = "I can see the the red rose rose and the purple lilac."
print(text)
for m in re.finditer(r"\b(\w+)\s+\1\b", text):
    print('-' * m.start() + '^' * (m.end() - m.start()))

* Note: Using groups in `re.findall()` changes the output type from a list of full matches to a list of captures group tuples

In [ ]:
re.findall(r"the\s+(?P<color>\w+)\s+(?P<plant>\w+)\b", "I can see the red rose and the purple lilac.")

In [ ]:
re.findall(r"the\s+\w+\s+\w+\b", "I can see the red rose and the purple lilac.")

### Greedy vs lazy matching
* Greedy matching
    * The default behavior in regular expressions
    * Matches as much text as possible before letting the rest of the pattern attempt to match
* Lazy (non-greedy) matching
    * Flips this behavior
    * Matches as little text as possible, expanding one character at a time only when the rest of the pattern fails.
* Let's look at a couple of examples
    * `*` - Match preceding character any number of times
    * `?` - Match preceding character zero or one time
    * `.` - Matches any character
* Appending `?` to any standard quantifier converts it from greedy to lazy

In [ ]:
html_str = "<b>hello</b> and <b>world</b>"
html_str

In [ ]:
# Greedy: grabs from the first <b> to the LAST </b>
greedy_match = re.findall(r"<b>.*</b>", html_str)
print("Greedy:", greedy_match)

In [ ]:
# Lazy: grabs from first <b> to the NEXT </b>
lazy_match = re.findall(r"<b>.*?</b>", html_str)
print("Lazy:  ", lazy_match)

|Description|Greedy|Lazy|Behavior|
|-:|-:|-:|:-|
|0 or more|`*`|`*?`|Greedy eats all; Lazy takes 0 first|
|1 or more|`+`|`+?`|Greedy eats all; Lazy takes 1 first|
|0 or 1|`?`|`??`|Greedy takes 1; Lazy takes 0|
|Range|`{n,m}`|`{n,m}?`|Greedy tries m; Lazy tries n|

### Flags
* By adding [flags](https://docs.python.org/3/howto/regex.html#compilation-flags) to the functions you can change the way the matches work
* `re.ASCII` or `re.A`: Makes several escapes like \w, \b, \s and \d match only on ASCII characters.
* `re.DOTALL`, `re.S`: Make `.` match any character, including newlines.
* `re.IGNORECASE`, `re.I`: Do case-insensitive matches.
* `re.MULTILINE`, `re.M`: Multi-line matching, affecting `^` and `$`.
    * `^` indicates the start of a line
    * `$` indicated the end of a line
* `re.VERBOSE`, `re.X`: Enable verbose REs, which can be organized more cleanly and understandably.

In [ ]:
re.findall(r"the", "The beauty and the beast", flags=re.IGNORECASE)

In [ ]:
date_text = "2026-10-10"

# Without re.VERBOSE (Hard to read and maintain)
dense_pattern = r"^(19|20)\d{2}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])$"

# With re.VERBOSE (Self-documenting and organized)
verbose_pattern = re.compile(
    r"""
    ^                            # Start of string
    (?P<year> (?:19|20)\d{2} )   # 4-digit year (1900-2099)
    -                            # Literal dash separator
    (?P<month> 0[1-9]|1[0-2] )   # 2-digit month (01-12)
    -                            # Literal dash separator
    (?P<day> 0[1-9]|[12]\d|3[01] )# 2-digit day (01-31)
    $                            # End of string
""",
    re.VERBOSE,
)

match = verbose_pattern.search(date_text)
print(match.groupdict())

* `re.VERBOSE` is useful when you write complex patterns

### An overview of the most used symbols in patterns

| Symbol | Description |
|-:|:-|
|`.`| Matches any single character except newline |
|`*`| Match preceding character any number of times |
|`+`| Match preceding character one or more times |
|`?`| Match preceding character zero or one time |
|`{n}`| Match preceding character exactly n times |
|`{n, m}`| Match preceding character at least n, at most m times |
|`[abc]`| Match any of the characters in the brackets |
|`[^abc]`| Match any character not given in the brackets 
|`[a-z]`| Match any character from a-z |
|`\b`| Match word boundary |
|`\d`| Match digit |
|`\s`| Match whitespace |
|`\w`| Match word character (`[A-Za-z0-9_]`<br> and all characters defined as letters by Unicode) |
|`^` or `\A`| Matches beginning of input |
|`$` or `\Z`| Matches end of input |
|`()`| Group, can be referenced later |
|`(?P<name>)` | Named group |
| `\1`, `\2`, ... | Reference to group defined earlier |
| `\(, \)` | Required to include parentheses in pattern |

Now we can re-visit the motivational example we started with:

### Email Regex Pattern Breakdown

**Pattern:** `r"[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}"`

```text
  [\w.-]+  @  [\w.-]+  \.  [a-zA-Z]{2,}
  │        │  │        │   │
  │        │  │        │   └── Top-Level Domain (TLD): 2+ letters (e.g., com, org, io)
  │        │  │        └────── Literal Dot: Escaped (\.) because . means "any character"
  │        │  └─────────────── Domain Name: 1+ word characters, dots, or hyphens
  │        └────────────────── Literal '@' symbol
  └─────────────────────────── Username / Local Part: 1+ word characters, dots, or hyphens
```

* `[\w.-]+` (Username / local part)
  * `[...]` — Character class matching any single character inside the brackets.
  * `\w` — Shorthand for word characters (a-z, A-Z, 0-9, _).
  * `.` and `-` — Matches a literal dot or hyphen (inside character classes [...], the dot automatically loses its special "match everything" power).
  * `+` — Quantifier for 1 or more characters.
* `@` (Literal Separator)
  * Matches the single `@` character strictly once.
* `[\w.-]+` (domain name)
  * One or more word characters, dots, or hyphens (handles subdomains like dev-labs or mail.techcorp).
* `\.` (domain separator)
  * Escaped backslash `\.` forces a literal dot. Outside of square brackets, an unescaped dot `.` matches any character (which would accidentally match user@domainXcom).
* `[a-zA-Z]{2,}` (top-level domain / TLD)
  * `[a-zA-Z]` — Restricts TLDs to uppercase and lowercase letters only (excluding digits or underscores).
  * `{2,}` — Quantifier requiring at least 2 characters (e.g., .io, .com, .museum).

* Note: This pattern does not capture all possible email addresses, the full pattern for that would be very long and complex.

### Phone Number Regex Pattern Breakdown

**Pattern:** `r"\+?\d[\d.-]{8,}\d"`

```text
  \+?   \d   [\d.-]{8,}   \d
  │     │    │            │
  │     │    │            └── Trailing Digit: Ensures sequence ends on a digit (ignores '!' or '.')
  │     │    └────────────── Middle Sequence: 8+ characters of digits, dots, or hyphens
  │     └────────────────── Leading Digit: Guarantees sequence starts with a number
  └──────────────────────── Optional Plus Sign: Matches '+' for international codes (0 or 1 time)
```

* `\+?` (Optional Country Code Indicator)
  * `\+` — Escaped plus sign (since `+` is normally a quantifier meaning "1 or more").
  * `?` — Quantifier meaning 0 or 1 time (allows numbers with or without a leading +).
* `\d` (Leading Digit)
  * Shorthand for any digit (0-9). Ensures the number starts with a digit immediately after the optional `+`.
* `[\d.-]{8,}` (Middle Digits & Separators)
  * `[...]` — Character class matching digits (`\d`), literal dots (`.`), or hyphens (`-`).
  * `{8,}` — Quantifier requiring at least 8 of these characters in sequence.
* `\d` (Trailing Digit)
  * Requires the final character to be a digit (0-9). This prevents trailing sentence punctuation (like call 555-123-4567!) from being incorrectly captured inside the phone match.

### When not to regex?
* Sometimes your problem will not need a full regex solution
* `str.split()` will split a string into words
* Use `str.startswith()` or `str.endsswith()` to check the start or end of a string